# 第14章　マルチモーダル医療AI ― 画像だけで診ない

**『医療診断支援AI開発　社会実装編 ― 臨床現場に届ける（社会実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-social

## 融合の3つの方式 ― early / late / cross-attention

In [ ]:
import torch

# 後期融合の最小形
z_img = cnn(image)              # 画像 -> 256次元ベクトル
z_tab = mlp(tabular)            # 検査値・年齢 -> 32次元
z_txt = text_encoder(note)     # 所見テキスト -> 128次元
z = torch.cat([z_img, z_tab, z_txt], dim=1)  # 連結
logit = classifier(z)          # まとめて判断

## 時間軸を持つマルチモーダル ― 縦断データと画像

In [ ]:
per_visit = [encode(img_t, tab_t) for img_t, tab_t in visits]   # 各受診→ベクトル
dt = time_gaps(visit_dates)                                     # 前回からの経過日数
seq = [v + time_embed(t) for v, t in zip(per_visit, dt)]        # 経過時間を注入
risk = temporal_transformer(seq)[-1]                            # 最新時点の予後スコア

## 粗いラベルで、細かく学ぶ ― 多重インスタンス学習（MIL）

In [ ]:
import torch

# attention-based MIL：各パッチ特徴に注意重みを付け、袋の表現へ集約
H = encoder(patches)                          # (パッチ数, d) 各インスタンスの特徴
a = torch.softmax(attn_v(torch.tanh(attn_u(H))), dim=0)  # 各パッチの注意重み（合計1）
z = (a * H).sum(0)                            # 袋（検査/スライド）全体の表現
logit = classifier(z)                         # 袋ラベルだけで学習

## 関係を明示的に扱う ― グラフニューラルネットワーク

In [ ]:
import torch

# 集団グラフの一層（GCN風）：患者=ノード、類似=エッジ
H = image_features                      # (患者数, 特徴次元)
A_hat = normalize_adj(similarity_graph) # 自己ループ付き正規化隣接行列
H = torch.relu(A_hat @ H @ W)           # 隣人の特徴を集約して更新
logits = H @ W_out                      # 各患者ノードの分類